# TFM V4 — final full-encoder adaptation

V4 is the last TFM transfer test. It reuses the V1 tokens, replaces the old clinical head, and fine-tunes the complete official MTP-pretrained TFM encoder for ZuCo sentiment. The tokenizer stays frozen.

```text
V1 token maps -> sample one reader per training sentence/epoch
              -> fully trainable official MTP encoder + new sentiment head
              -> average every validation/test reader's probabilities
              -> aligned vs separately trained shuffled control -> final gate
```

| Fixed choice | V4 value |
| --- | --- |
| Tokenizer | Frozen; reuse `tokens_v1` |
| Encoder | Official TFM 64x4 MTP checkpoint; every floating-point weight trainable |
| Montage adapter | Six 16-channel groups plus one 8-channel tail; trainable group-aware mixer |
| Training sampling | One reader per sentence per epoch; reader resampled each epoch |
| Validation/test | Probabilities averaged across all available readers |
| Splits | 5 unseen-sentence folds, seeds 42/52/62 |
| Control | A separately trained, within-split, no-fixed-point shuffled model |
| Resumption | Saves every completed setup/fold; a disconnect loses at most one fit |
| Decision | Final five-part gate with a four-version-corrected 98.75% bootstrap interval |


## Run instructions

Select **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. Cell 4 is the long cell because it trains 30 models: aligned and shuffled for 5 folds and 3 seeds. Progress and completed fits are saved in Drive. If Colab disconnects, reconnect and run all cells again; completed fits are reused.


In [ ]:
# 1) Fetch this codebase and install only missing official-encoder dependencies.
from pathlib import Path
import importlib.metadata, importlib.util, os, subprocess, sys

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
else:
    run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)
requirements = {
    "einops": "einops==0.8.0",
    "linear_attention_transformer": "linear-attention-transformer==0.19.1",
    "timm": "timm==1.0.14",
}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    run([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Official-encoder dependencies already available")
project_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
os.chdir(PROJECT_ROOT / "tfm")
print("Working directory:", Path.cwd())
print("Project revision:", project_revision)


In [ ]:
# 2) Mount Drive and use the established Data / CachedArtifacts / Results layout.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
CACHE_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/tfm"
RESULTS_ROOT = THESIS_ROOT / "Results/eeg_tokenizer/tfm"
TOKEN_CACHE = CACHE_ROOT / "tokens_v1"
PACKED_TOKEN_CACHE = CACHE_ROOT / "token_records_v2_packed"
CHECKPOINT_CACHE = CACHE_ROOT / "upstream_checkpoints/huggingface/pretrained"
RESULTS_DIR = RESULTS_ROOT / "encoder_finetune_v4"

if not TOKEN_CACHE.exists():
    raise FileNotFoundError(f"V1 token cache not found: {TOKEN_CACHE}")
for path in (PACKED_TOKEN_CACHE, CHECKPOINT_CACHE, RESULTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
print("Token cache:", TOKEN_CACHE)
print("Packed token cache:", PACKED_TOKEN_CACHE)
print("V4 results:", RESULTS_DIR)


In [ ]:
# 3) Pin the official source and reuse/download the verified 5.1 MiB MTP checkpoint.
import hashlib, urllib.request

UPSTREAM_URL = "https://github.com/Jathurshan0330/TFM-Tokenizer.git"
UPSTREAM_REVISION = "2d6da482b16dabbb2ebec808fa9f505fc6f367c4"
UPSTREAM_ROOT = Path("/content/TFM-Tokenizer")
environment = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
if not UPSTREAM_ROOT.exists():
    run(["git", "clone", "--depth", "1", UPSTREAM_URL, str(UPSTREAM_ROOT)], env=environment)
run(["git", "fetch", "--depth", "1", "origin", UPSTREAM_REVISION], cwd=UPSTREAM_ROOT, env=environment)
run(["git", "checkout", "--detach", UPSTREAM_REVISION], cwd=UPSTREAM_ROOT, env=environment)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_ROOT, text=True).strip()
if actual_revision != UPSTREAM_REVISION:
    raise RuntimeError(f"Official source revision mismatch: {actual_revision}")

ENCODER_CHECKPOINT = CHECKPOINT_CACHE / "tfm_encoder_mtp_last.pth"
ENCODER_SIZE = 5_308_635
ENCODER_SHA256 = "738d56a2021dd363c9d21c4804c441b486b28e33f73883dcf49623f9db8ad973"
ENCODER_URL = (
    "https://huggingface.co/Jathurshan/TFM-Tokenizer/resolve/"
    "e63f37850348b2c17429b5797442c0888163e4c9/"
    "pretrained/tfm_encoder_mtp_last.pth?download=true"
)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(2**20):
            digest.update(chunk)
    return digest.hexdigest()

checkpoint_valid = (
    ENCODER_CHECKPOINT.exists()
    and ENCODER_CHECKPOINT.stat().st_size == ENCODER_SIZE
    and file_sha256(ENCODER_CHECKPOINT) == ENCODER_SHA256
)
if not checkpoint_valid:
    temporary = ENCODER_CHECKPOINT.with_suffix(".download")
    print(f"Downloading official MTP encoder ({ENCODER_SIZE / 2**20:.1f} MiB)")
    urllib.request.urlretrieve(ENCODER_URL, temporary)
    if temporary.stat().st_size != ENCODER_SIZE or file_sha256(temporary) != ENCODER_SHA256:
        raise IOError("Downloaded MTP encoder failed size/SHA-256 verification")
    temporary.replace(ENCODER_CHECKPOINT)
else:
    print("Reusing verified Drive-cached MTP encoder")
print("Encoder checkpoint:", ENCODER_CHECKPOINT)
print("Official source revision:", actual_revision)


In [ ]:
# 4) Verify gradients, then run/resume the final fine-tuning evaluation.
import importlib, json
import numpy as np
import torch
import src.finetune_probe as finetune_probe_module
importlib.reload(finetune_probe_module)
from src.finetune_probe import (
    FinetunableOfficialTFMEncoder,
    FinetuneProbeConfig,
    evaluate_finetuned_encoder,
)
from src.token_map import LABEL_TO_INDEX, load_or_pack_token_records

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU, then rerun")

def save_runtime_stage(stage, **details):
    payload = {"stage": stage, **details}
    temporary = RESULTS_DIR / "runtime_status.tmp.json"
    temporary.write_text(json.dumps(payload, indent=2))
    temporary.replace(RESULTS_DIR / "runtime_status.json")
    print("Runtime stage:", stage, details)

config = FinetuneProbeConfig()
save_runtime_stage("v4_started")
records, token_metadata, token_report = load_or_pack_token_records(
    TOKEN_CACHE, PACKED_TOKEN_CACHE, workers=8
)
print("Loaded recordings/sentences/subjects:", token_report["n_recordings"], token_report["n_sentences"], token_report["n_subjects"])
save_runtime_stage("token_cache_loaded", recordings=len(records))

# One-recording forward/backward smoke test catches a detached encoder or bad head before 30 fits begin.
smoke_model = FinetunableOfficialTFMEncoder(
    UPSTREAM_ROOT, ENCODER_CHECKPOINT, config=config, device="cuda", initialization_seed=0
)
smoke_model.train()
smoke_record = records[0]
smoke_tokens = torch.as_tensor(smoke_record.tokens[None].astype(np.int64), device=smoke_model.device)
smoke_label = torch.as_tensor([LABEL_TO_INDEX[smoke_record.label]], device=smoke_model.device)
smoke_logits = smoke_model(smoke_tokens)
smoke_loss = torch.nn.functional.cross_entropy(smoke_logits, smoke_label)
smoke_loss.backward()
encoder_gradients = sum(int(parameter.grad is not None and bool(torch.isfinite(parameter.grad).all())) for parameter in smoke_model.encoder_parameters)
head_gradients = sum(int(parameter.grad is not None and bool(torch.isfinite(parameter.grad).all())) for parameter in smoke_model.head_parameters)
if smoke_logits.shape != (1, 3) or encoder_gradients == 0 or head_gradients == 0:
    raise RuntimeError("V4 smoke test did not produce valid encoder/head gradients")
print("Smoke logits/loss:", smoke_logits.detach().cpu().numpy(), float(smoke_loss.detach().cpu()))
print("Gradient-bearing encoder/head tensors:", encoder_gradients, head_gradients)
print("Encoder load/trainability report:", smoke_model.report)
del smoke_model, smoke_tokens, smoke_logits, smoke_loss
torch.cuda.empty_cache()
save_runtime_stage("gradient_smoke_test_passed")

runtime_environment = {
    "project_revision": project_revision,
    "official_source_revision": actual_revision,
    "checkpoint_sha256": file_sha256(ENCODER_CHECKPOINT),
    "packages": {
        name: importlib.metadata.version(name)
        for name in ("torch", "einops", "linear-attention-transformer", "timm", "scikit-learn")
    },
}
(RESULTS_DIR / "runtime_environment.json").write_text(json.dumps(runtime_environment, indent=2))
metrics, predictions, history, summary, delta, gate = evaluate_finetuned_encoder(
    records=records,
    repo_dir=UPSTREAM_ROOT,
    checkpoint_path=ENCODER_CHECKPOINT,
    output_dir=RESULTS_DIR,
    cache_report=token_report,
    source_revision=actual_revision,
    config=config,
    device="cuda",
    resume=True,
    status_callback=save_runtime_stage,
)
display(summary)
print("Corrected paired bootstrap:", delta)
print("Decision:", gate["decision"])
save_runtime_stage("evaluation_complete", decision=gate["decision"])


In [ ]:
# 5) Read the saved final result and create the seed comparison figure.
import json
import matplotlib.pyplot as plt
import pandas as pd

metrics = pd.read_csv(RESULTS_DIR / "fold_metrics.csv")
summary = pd.read_csv(RESULTS_DIR / "summary.csv", header=[0, 1])
gate = json.loads((RESULTS_DIR / "viability_gate.json").read_text())
display(summary)
seed_scores = metrics.groupby(["seed", "setup"])["macro_f1"].mean().unstack("setup")
display(seed_scores)
axes = seed_scores[["encoder_finetune", "encoder_finetune_shuffled", "majority"]].plot.bar(figsize=(8, 4))
axes.axhline(1 / 3, color="black", linestyle="--", linewidth=1, label="1/3 reference")
axes.set(title="TFM V4 macro-F1 by split seed", ylabel="macro-F1", xlabel="seed")
axes.legend(loc="best")
plt.tight_layout()
figure_path = RESULTS_DIR / "macro_f1_by_seed.png"
plt.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(json.dumps(gate, indent=2))
print("Saved figure:", figure_path)


## Interpretation boundary

V4 remains an unseen-sentence evaluation for the known ZuCo reader pool, not an unseen-subject test. It is the final attempt in this TFM branch. If the aligned fine-tuned encoder does not beat its separately trained shuffled control under the locked gate, there will be no V5 or post-hoc architecture/hyperparameter variation.
